# TalkFreeEngine_1.13

## Knowledge Object Schema Upgrade

This version prepares the Colab prototype for import into the future website.

## Updates in 1.13

- Converts each word into a full Knowledge Object
- Adds portable export schema
- Adds website-ready JSON export
- Adds object validation
- Adds schema migration
- Adds parent notes, mastery score, pronunciation, context, relationships, goals, and AI expansion rules
- Keeps child board
- Keeps Parent Insights
- Keeps Language Genome
- Keeps local-first profile data

## Core rule

Build the engine first.  
The website renders the engine.


In [1]:

# TalkFreeEngine_1.13 — Knowledge Object Schema Setup

from IPython.display import display, clear_output, Javascript
import ipywidgets as widgets
import json
from pathlib import Path
from datetime import datetime

APP_VERSION = "TalkFreeEngine_1.13"

DATA_DIR = Path(".")
TYPE_STYLES_FILE = DATA_DIR / "type_styles.json"
WORDS_FILE = DATA_DIR / "word_objects.json"
GRAPH_FILE = DATA_DIR / "language_graph.json"
STAGE_FILE = DATA_DIR / "stage_boards.json"
PROFILE_FILE = DATA_DIR / "child_profile.json"

sentence = []
parent_tools_open = False
last_deleted_word = None
last_deleted_payload = None
system_message = ""

DEFAULT_TYPE_STYLES = {
    "pronoun": {"color": "#0057B8", "label": "Pronoun", "emoji": "👤"},
    "verb": {"color": "#118C4F", "label": "Action", "emoji": "🏃"},
    "food": {"color": "#F57C00", "label": "Food/Drink", "emoji": "🍽️"},
    "feeling": {"color": "#C2185B", "label": "Feeling", "emoji": "😊"},
    "place": {"color": "#6A1B9A", "label": "Place", "emoji": "🏠"},
    "person": {"color": "#3949AB", "label": "Person", "emoji": "👨‍👩‍👧"},
    "utility": {"color": "#424242", "label": "Utility", "emoji": "🔤"},
    "safety": {"color": "#B00020", "label": "Safety", "emoji": "🆘"},
    "object": {"color": "#00796B", "label": "Object", "emoji": "🧸"},
    "social": {"color": "#AD1457", "label": "Social", "emoji": "💬"},
    "time": {"color": "#5D4037", "label": "Time", "emoji": "⏰"}
}

DEFAULT_WORDS = {
    # Stage 1 core
    "I": {"image": "👤", "type": "pronoun", "base_weight": 100, "stage": 1},
    "want": {"image": "🤲", "type": "verb", "base_weight": 100, "stage": 1},
    "need": {"image": "❗", "type": "verb", "base_weight": 95, "stage": 1},
    "help": {"image": "🆘", "type": "safety", "base_weight": 100, "stage": 1},
    "yes": {"image": "👍", "type": "utility", "base_weight": 90, "stage": 1},
    "no": {"image": "👎", "type": "utility", "base_weight": 90, "stage": 1},
    "more": {"image": "➕", "type": "utility", "base_weight": 90, "stage": 1},
    "all done": {"image": "✅", "type": "utility", "base_weight": 90, "stage": 1},
    "potty": {"image": "🚽", "type": "place", "base_weight": 100, "stage": 1},
    "eat": {"image": "🍽️", "type": "verb", "base_weight": 90, "stage": 1},
    "drink": {"image": "🥤", "type": "verb", "base_weight": 90, "stage": 1},
    "play": {"image": "🎮", "type": "verb", "base_weight": 85, "stage": 1},
    "go": {"image": "🚶", "type": "verb", "base_weight": 95, "stage": 1},
    "mom": {"image": "👩", "type": "person", "base_weight": 95, "stage": 1},
    "dad": {"image": "👨", "type": "person", "base_weight": 95, "stage": 1},
    "milk": {"image": "🥛", "type": "food", "base_weight": 85, "stage": 1},
    "water": {"image": "💧", "type": "food", "base_weight": 95, "stage": 1},
    "food": {"image": "🍽️", "type": "food", "base_weight": 95, "stage": 1},
    "hurt": {"image": "🤕", "type": "safety", "base_weight": 90, "stage": 1},
    "happy": {"image": "😊", "type": "feeling", "base_weight": 80, "stage": 1},
    "sad": {"image": "😢", "type": "feeling", "base_weight": 80, "stage": 1},
    "mad": {"image": "😡", "type": "feeling", "base_weight": 75, "stage": 1},
    "sleep": {"image": "😴", "type": "verb", "base_weight": 75, "stage": 1},
    "again": {"image": "🔁", "type": "utility", "base_weight": 82, "stage": 1},
    "break": {"image": "⏸️", "type": "utility", "base_weight": 76, "stage": 1},
    "hug": {"image": "🤗", "type": "social", "base_weight": 80, "stage": 1},
    "tablet": {"image": "📱", "type": "object", "base_weight": 78, "stage": 1},
    "toy": {"image": "🧸", "type": "object", "base_weight": 75, "stage": 1},

    # Stage 2 sentence building
    "you": {"image": "👉", "type": "pronoun", "base_weight": 80, "stage": 2},
    "we": {"image": "👥", "type": "pronoun", "base_weight": 75, "stage": 2},
    "am": {"image": "🧍", "type": "verb", "base_weight": 90, "stage": 2},
    "feel": {"image": "❤️", "type": "verb", "base_weight": 80, "stage": 2},
    "like": {"image": "👍", "type": "verb", "base_weight": 70, "stage": 2},
    "love": {"image": "💗", "type": "verb", "base_weight": 70, "stage": 2},
    "can": {"image": "✅", "type": "verb", "base_weight": 60, "stage": 2},
    "to": {"image": "➡️", "type": "utility", "base_weight": 100, "stage": 2},
    "jump": {"image": "🤸", "type": "verb", "base_weight": 85, "stage": 2},
    "run": {"image": "🏃", "type": "verb", "base_weight": 75, "stage": 2},
    "walk": {"image": "🚶", "type": "verb", "base_weight": 70, "stage": 2},
    "see": {"image": "👀", "type": "verb", "base_weight": 70, "stage": 2},
    "have": {"image": "✋", "type": "verb", "base_weight": 70, "stage": 2},
    "wash": {"image": "🧼", "type": "verb", "base_weight": 70, "stage": 2},
    "brush": {"image": "🪥", "type": "verb", "base_weight": 68, "stage": 2},
    "read": {"image": "📚", "type": "verb", "base_weight": 70, "stage": 2},
    "watch": {"image": "👀", "type": "verb", "base_weight": 72, "stage": 2},
    "bathroom": {"image": "🚽", "type": "place", "base_weight": 95, "stage": 2},
    "home": {"image": "🏠", "type": "place", "base_weight": 80, "stage": 2},
    "outside": {"image": "🌳", "type": "place", "base_weight": 85, "stage": 2},
    "school": {"image": "🏫", "type": "place", "base_weight": 70, "stage": 2},
    "bed": {"image": "🛏️", "type": "place", "base_weight": 70, "stage": 2},
    "park": {"image": "🛝", "type": "place", "base_weight": 65, "stage": 2},
    "juice": {"image": "🧃", "type": "food", "base_weight": 80, "stage": 2},
    "snack": {"image": "🍪", "type": "food", "base_weight": 75, "stage": 2},
    "chicken nuggets": {"image": "🍗", "type": "food", "base_weight": 80, "stage": 2},
    "pizza": {"image": "🍕", "type": "food", "base_weight": 70, "stage": 2},
    "apple": {"image": "🍎", "type": "food", "base_weight": 60, "stage": 2},
    "banana": {"image": "🍌", "type": "food", "base_weight": 60, "stage": 2},
    "yogurt": {"image": "🥣", "type": "food", "base_weight": 68, "stage": 2},
    "cracker": {"image": "🍪", "type": "food", "base_weight": 68, "stage": 2},
    "jumpy": {"image": "🤸", "type": "object", "base_weight": 100, "stage": 2},
    "trampoline": {"image": "🤸", "type": "object", "base_weight": 85, "stage": 2},
    "please": {"image": "🙏", "type": "social", "base_weight": 70, "stage": 2},
    "thank you": {"image": "🙏", "type": "social", "base_weight": 65, "stage": 2},
    "me": {"image": "👤", "type": "pronoun", "base_weight": 60, "stage": 2},
    "okay": {"image": "👌", "type": "feeling", "base_weight": 75, "stage": 2},
    "hungry": {"image": "🍽️", "type": "feeling", "base_weight": 90, "stage": 2},
    "thirsty": {"image": "🥤", "type": "feeling", "base_weight": 90, "stage": 2},
    "sleepy": {"image": "😴", "type": "feeling", "base_weight": 85, "stage": 2},
    "tired": {"image": "🥱", "type": "feeling", "base_weight": 80, "stage": 2},
    "sick": {"image": "🤢", "type": "feeling", "base_weight": 80, "stage": 2},
    "scared": {"image": "😨", "type": "feeling", "base_weight": 75, "stage": 2},
    "belly hurts": {"image": "🤢", "type": "safety", "base_weight": 70, "stage": 2},

    # Stage 3 fluent language
    "he": {"image": "👦", "type": "pronoun", "base_weight": 50, "stage": 3},
    "she": {"image": "👧", "type": "pronoun", "base_weight": 50, "stage": 3},
    "they": {"image": "👥", "type": "pronoun", "base_weight": 45, "stage": 3},
    "my": {"image": "👤", "type": "pronoun", "base_weight": 55, "stage": 3},
    "your": {"image": "👉", "type": "pronoun", "base_weight": 48, "stage": 3},
    "what": {"image": "❓", "type": "utility", "base_weight": 60, "stage": 3},
    "where": {"image": "📍", "type": "utility", "base_weight": 60, "stage": 3},
    "who": {"image": "👤", "type": "utility", "base_weight": 52, "stage": 3},
    "why": {"image": "❓", "type": "utility", "base_weight": 48, "stage": 3},
    "when": {"image": "⏰", "type": "time", "base_weight": 48, "stage": 3},
    "how": {"image": "❓", "type": "utility", "base_weight": 46, "stage": 3},
    "now": {"image": "⏰", "type": "time", "base_weight": 65, "stage": 3},
    "later": {"image": "⏳", "type": "time", "base_weight": 60, "stage": 3},
    "frustrated": {"image": "😤", "type": "feeling", "base_weight": 58, "stage": 3},
}

DEFAULT_STAGE_BOARDS = {
    "Stage 1: Emerging Communicator": ["I", "want", "need", "help", "yes", "no", "more", "all done", "potty"],
    "Stage 2: Sentence Builder": [
        "I", "you", "we", "want", "need", "am", "feel", "like",
        "love", "can", "to", "go", "eat", "drink", "play", "jump",
        "help", "stop", "more", "all done", "yes", "no", "mom", "dad",
        "potty", "bathroom", "outside", "home", "food", "water", "milk", "snack",
        "happy", "sad", "mad", "hurt"
    ],
    "Stage 3: Fluent Builder": [
        "I", "you", "we", "he", "she", "they", "my", "your",
        "want", "need", "am", "feel", "like", "love", "can", "have",
        "to", "go", "eat", "drink", "play", "jump", "run", "walk",
        "see", "watch", "read", "wash", "brush", "help", "stop", "more",
        "what", "where", "who", "why", "when", "how", "now", "later"
    ]
}

DEFAULT_GRAPH = {
    "I": {"want": 120, "need": 115, "am": 105, "feel": 90, "like": 80, "love": 78, "can": 70, "have": 65, "see": 60},
    "I want": {"to": 140, "food": 90, "drink": 88, "milk": 86, "water": 84, "snack": 80, "more": 78, "mom": 70, "dad": 70, "outside": 68, "play": 66, "potty": 62, "tablet": 60},
    "I want to": {"go": 135, "eat": 120, "drink": 110, "play": 108, "jump": 105, "run": 88, "walk": 84, "see": 78, "have": 75, "sleep": 72, "watch": 68, "read": 62},
    "I want to go": {"potty": 150, "bathroom": 130, "outside": 105, "home": 95, "school": 80, "bed": 75, "park": 68},
    "I want to eat": {"food": 120, "snack": 110, "chicken nuggets": 105, "pizza": 95, "apple": 78, "banana": 76, "yogurt": 74, "cracker": 72},
    "I want to drink": {"water": 125, "milk": 115, "juice": 105, "please": 70, "more": 66},
    "I want to play": {"jumpy": 120, "trampoline": 115, "outside": 95, "toy": 90, "tablet": 80},
    "I want to jump": {"jumpy": 140, "trampoline": 120, "outside": 80, "again": 75, "more": 65},
    "I need": {"to": 140, "help": 120, "potty": 95, "water": 88, "food": 86, "milk": 84, "mom": 82, "dad": 82, "break": 80, "hug": 74, "sleep": 72},
    "I need to": {"go": 150, "eat": 105, "drink": 100, "sleep": 92, "wash": 78, "brush": 74, "play": 66, "see": 60},
    "I need to go": {"potty": 160, "bathroom": 140, "home": 95, "outside": 85, "bed": 70, "school": 64},
    "I am": {"happy": 110, "sad": 105, "mad": 102, "scared": 100, "hurt": 118, "hungry": 110, "thirsty": 108, "sleepy": 106, "tired": 98, "sick": 95, "okay": 90},
    "I feel": {"happy": 105, "sad": 102, "mad": 100, "scared": 98, "hurt": 118, "hungry": 105, "thirsty": 102, "sleepy": 98, "tired": 95, "sick": 90},
    "I like": {"jumpy": 115, "trampoline": 100, "play": 95, "food": 80, "drink": 78, "outside": 76},
    "I love": {"you": 140, "mom": 120, "dad": 120, "play": 70, "jumpy": 68},
    "can": {"I": 110, "you": 90, "we": 80},
    "can I": {"go": 110, "play": 100, "eat": 90, "drink": 88, "have": 86, "see": 75, "help": 70},
    "can I have": {"milk": 120, "water": 115, "juice": 105, "snack": 90, "toy": 82, "tablet": 75, "please": 70},
    "help": {"me": 120, "please": 110, "mom": 100, "dad": 100, "potty": 90, "open": 80},
    "more": {"please": 110, "food": 100, "drink": 98, "milk": 96, "water": 92, "play": 90, "jumpy": 88},
    "drink": {"water": 125, "milk": 115, "juice": 105, "please": 75, "more": 70},
    "eat": {"food": 115, "snack": 105, "chicken nuggets": 100, "pizza": 95, "apple": 80, "banana": 78, "yogurt": 76},
    "go": {"potty": 130, "bathroom": 115, "outside": 105, "home": 95, "school": 82, "park": 72, "bed": 70},
    "where": {"mom": 110, "dad": 110, "toy": 90, "tablet": 85, "jumpy": 80, "bathroom": 75},
}

DEFAULT_PROFILE = {
    "selected_stage": "Stage 2: Sentence Builder",
    "word_counts": {},
    "phrase_counts": {},
    "tap_history": [],
    "sentence_history": [],
    "mission": "Basic communication stays free. The child never hits a wall."
}

def save_json(path, data):
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")

def load_json(path, default):
    if not path.exists():
        save_json(path, default)
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        save_json(path, default)
        return default

TYPE_STYLES = load_json(TYPE_STYLES_FILE, DEFAULT_TYPE_STYLES)
WORDS = load_json(WORDS_FILE, DEFAULT_WORDS)
GRAPH = load_json(GRAPH_FILE, DEFAULT_GRAPH)
STAGE_BOARDS = load_json(STAGE_FILE, DEFAULT_STAGE_BOARDS)
PROFILE = load_json(PROFILE_FILE, DEFAULT_PROFILE)

# Merge defaults into old files if they already exist in Colab.
for k, v in DEFAULT_WORDS.items():
    WORDS.setdefault(k, v)
for k, v in DEFAULT_GRAPH.items():
    GRAPH.setdefault(k, {})
    GRAPH[k].update(v)
for k, v in DEFAULT_STAGE_BOARDS.items():
    STAGE_BOARDS[k] = v
for k, v in DEFAULT_TYPE_STYLES.items():
    TYPE_STYLES.setdefault(k, v)


PROFILE.setdefault("engine_version", APP_VERSION)
PROFILE.setdefault("strict_prediction_mode", True)
PROFILE.setdefault("quick_phrases_enabled", True)

def set_engine_version():
    PROFILE["engine_version"] = APP_VERSION

def safe_asset_path(word, word_type):
    clean = word.lower().replace(" ", "_").replace("'", "")
    return "assets/" + word_type + "/" + clean + ".png"

def repair_core_data():
    required_words = {
        "I": {"image": "👤", "type": "pronoun", "base_weight": 130, "stage": 1},
        "want": {"image": "🤲", "type": "verb", "base_weight": 130, "stage": 1},
        "you": {"image": "👉", "type": "pronoun", "base_weight": 95, "stage": 1},
        "love": {"image": "💗", "type": "social", "base_weight": 115, "stage": 1},
        "I love you": {"image": "💗", "type": "social", "base_weight": 135, "stage": 1},
        "yes": {"image": "👍", "type": "utility", "base_weight": 120, "stage": 1},
        "no": {"image": "👎", "type": "utility", "base_weight": 120, "stage": 1},
        "please": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "thank you": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "no thank you": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "yes please": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "help please": {"image": "🆘", "type": "safety", "base_weight": 110, "stage": 1},
        "water": {"image": "💧", "type": "food", "base_weight": 120, "stage": 1},
        "apple juice": {"image": "🧃", "type": "food", "base_weight": 120, "stage": 1},
        "chicken nuggets": {"image": "🍗", "type": "food", "base_weight": 115, "stage": 1},
        "french fries": {"image": "🍟", "type": "food", "base_weight": 110, "stage": 1},
        "gummies": {"image": "🍬", "type": "food", "base_weight": 110, "stage": 1},
        "I'm hungry": {"image": "🍽️", "type": "feeling", "base_weight": 115, "stage": 1},
        "I'm thirsty": {"image": "🥤", "type": "feeling", "base_weight": 115, "stage": 1},
        "happy": {"image": "😊", "type": "feeling", "base_weight": 100, "stage": 1},
        "sad": {"image": "😢", "type": "feeling", "base_weight": 95, "stage": 1},
        "mad": {"image": "😡", "type": "feeling", "base_weight": 95, "stage": 1},
        "hurt": {"image": "🤕", "type": "safety", "base_weight": 100, "stage": 1},
        "okay": {"image": "👌", "type": "feeling", "base_weight": 100, "stage": 1},
        "good": {"image": "👍", "type": "feeling", "base_weight": 90, "stage": 1},
        "awesome": {"image": "🤩", "type": "feeling", "base_weight": 95, "stage": 1},
    }

    for word, data in required_words.items():
        WORDS[word] = data

    STAGE_BOARDS["Stage 1: Emerging Communicator"] = [
        "I", "want", "yes",
        "no", "please", "water",
        "apple juice", "chicken nuggets", "I love you"
    ]

    stage2_required = [
        "I", "want", "you", "love", "I love you", "please", "thank you",
        "yes", "no", "yes please", "no thank you", "help please",
        "water", "apple juice", "milk", "gummies", "french fries", "chicken nuggets",
        "I'm hungry", "I'm thirsty", "happy", "sad", "mad", "hurt", "okay", "good", "awesome",
        "time", "for", "bed", "potty", "go", "eat", "drink", "play", "mom", "dad", "help", "all done"
    ]
    STAGE_BOARDS["Stage 2: Sentence Builder"] = list(dict.fromkeys(stage2_required + STAGE_BOARDS.get("Stage 2: Sentence Builder", [])))

    protected_graph = {
        "I": {"love": 160, "want": 150, "need": 120, "am": 115, "feel": 110},
        "I love": {"you": 170, "mom": 110, "dad": 110},
        "I love you": {"thank you": 90, "mom": 80, "dad": 80},
        "I want": {
            "apple juice": 145, "water": 140, "chicken nuggets": 135,
            "french fries": 130, "gummies": 125, "milk": 120,
            "please": 110, "to": 100, "food": 95
        },
        "I am": {"hungry": 135, "thirsty": 130, "okay": 125, "good": 120, "awesome": 115, "happy": 110, "sad": 100, "mad": 98, "hurt": 105},
        "I feel": {"hungry": 135, "thirsty": 130, "okay": 125, "good": 120, "awesome": 115, "happy": 110, "sad": 100, "mad": 98, "hurt": 105},
        "yes": {"yes please": 135, "please": 125, "awesome": 95, "good": 90, "I love you": 80},
        "no": {"no thank you": 135, "thank you": 125, "stop": 95, "all done": 90},
        "help": {"help please": 145, "please": 130, "me": 120, "mom": 100, "dad": 100},
        "I'm hungry": {"food": 130, "chicken nuggets": 125, "french fries": 115, "please": 100},
        "I'm thirsty": {"water": 130, "apple juice": 125, "milk": 110, "please": 100},
    }

    for phrase, next_words in protected_graph.items():
        GRAPH.setdefault(phrase, {})
        GRAPH[phrase].update(next_words)

    set_engine_version()
    PROFILE["strict_prediction_mode"] = True


# -----------------------------
# TalkFreeEngine_1.10 migration
# -----------------------------
PROFILE.setdefault("engine_version", APP_VERSION)
PROFILE.setdefault("strict_prediction_mode", True)
PROFILE.setdefault("quick_phrases_enabled", True)

def set_engine_version():
    PROFILE["engine_version"] = APP_VERSION

def repair_core_data():
    # Protect against stale JSON files from older Colab runs.
    required_words = {
        "I": {"image": "👤", "type": "pronoun", "base_weight": 130, "stage": 1},
        "want": {"image": "🤲", "type": "verb", "base_weight": 130, "stage": 1},
        "you": {"image": "👉", "type": "pronoun", "base_weight": 95, "stage": 1},
        "love": {"image": "💗", "type": "social", "base_weight": 115, "stage": 1},
        "I love you": {"image": "💗", "type": "social", "base_weight": 135, "stage": 1},
        "yes": {"image": "👍", "type": "utility", "base_weight": 120, "stage": 1},
        "no": {"image": "👎", "type": "utility", "base_weight": 120, "stage": 1},
        "please": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "thank you": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "no thank you": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "yes please": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "help please": {"image": "🆘", "type": "safety", "base_weight": 110, "stage": 1},
        "water": {"image": "💧", "type": "food", "base_weight": 120, "stage": 1},
        "apple juice": {"image": "🧃", "type": "food", "base_weight": 120, "stage": 1},
        "chicken nuggets": {"image": "🍗", "type": "food", "base_weight": 115, "stage": 1},
        "french fries": {"image": "🍟", "type": "food", "base_weight": 110, "stage": 1},
        "gummies": {"image": "🍬", "type": "food", "base_weight": 110, "stage": 1},
        "I'm hungry": {"image": "🍽️", "type": "feeling", "base_weight": 115, "stage": 1},
        "I'm thirsty": {"image": "🥤", "type": "feeling", "base_weight": 115, "stage": 1},
        "happy": {"image": "😊", "type": "feeling", "base_weight": 100, "stage": 1},
        "sad": {"image": "😢", "type": "feeling", "base_weight": 95, "stage": 1},
        "mad": {"image": "😡", "type": "feeling", "base_weight": 95, "stage": 1},
        "hurt": {"image": "🤕", "type": "safety", "base_weight": 100, "stage": 1},
        "okay": {"image": "👌", "type": "feeling", "base_weight": 100, "stage": 1},
        "good": {"image": "👍", "type": "feeling", "base_weight": 90, "stage": 1},
        "awesome": {"image": "🤩", "type": "feeling", "base_weight": 95, "stage": 1},
    }

    for word, data in required_words.items():
        WORDS[word] = data

    # Stage 1 stays small but useful. Direct phrase included.
    STAGE_BOARDS["Stage 1: Emerging Communicator"] = [
        "I", "want", "yes",
        "no", "please", "water",
        "apple juice", "chicken nuggets", "I love you"
    ]

    # Make sure Stage 2 has emotional/social layer.
    stage2_required = [
        "I", "want", "you", "love", "I love you", "please", "thank you",
        "yes", "no", "yes please", "no thank you", "help please",
        "water", "apple juice", "milk", "gummies", "french fries", "chicken nuggets",
        "I'm hungry", "I'm thirsty", "happy", "sad", "mad", "hurt", "okay", "good", "awesome",
        "time", "for", "bed", "potty", "go", "eat", "drink", "play", "mom", "dad", "help", "all done"
    ]
    STAGE_BOARDS["Stage 2: Sentence Builder"] = list(dict.fromkeys(stage2_required + STAGE_BOARDS.get("Stage 2: Sentence Builder", [])))

    protected_graph = {
        "I": {"love": 160, "want": 150, "need": 120, "am": 115, "feel": 110},
        "I love": {"you": 170, "mom": 110, "dad": 110},
        "I love you": {"thank you": 90, "mom": 80, "dad": 80},
        "I want": {
            "apple juice": 145, "water": 140, "chicken nuggets": 135,
            "french fries": 130, "gummies": 125, "milk": 120,
            "please": 110, "to": 100, "food": 95
        },
        "I am": {"hungry": 135, "thirsty": 130, "okay": 125, "good": 120, "awesome": 115, "happy": 110, "sad": 100, "mad": 98, "hurt": 105},
        "I feel": {"hungry": 135, "thirsty": 130, "okay": 125, "good": 120, "awesome": 115, "happy": 110, "sad": 100, "mad": 98, "hurt": 105},
        "yes": {"yes please": 135, "please": 125, "awesome": 95, "good": 90, "I love you": 80},
        "no": {"no thank you": 135, "thank you": 125, "stop": 95, "all done": 90},
        "help": {"help please": 145, "please": 130, "me": 120, "mom": 100, "dad": 100},
        "I'm hungry": {"food": 130, "chicken nuggets": 125, "french fries": 115, "please": 100},
        "I'm thirsty": {"water": 130, "apple juice": 125, "milk": 110, "please": 100},
    }

    for phrase, next_words in protected_graph.items():
        GRAPH.setdefault(phrase, {})
        GRAPH[phrase].update(next_words)

    set_engine_version()
    PROFILE["strict_prediction_mode"] = True


def save_all():
    save_json(TYPE_STYLES_FILE, TYPE_STYLES)
    save_json(WORDS_FILE, WORDS)
    save_json(GRAPH_FILE, GRAPH)
    save_json(STAGE_FILE, STAGE_BOARDS)
    save_json(PROFILE_FILE, PROFILE)

def current_phrase():
    return " ".join(sentence).strip()

def selected_stage():
    return PROFILE.get("selected_stage", "Stage 2: Sentence Builder")

def stage_number():
    stage = selected_stage()
    if stage.startswith("Stage 1"):
        return 1
    if stage.startswith("Stage 2"):
        return 2
    return 3

def word_stage(word):
    return int(WORDS.get(word, {}).get("stage", 2))

def is_word_allowed_for_stage(word):
    return word_stage(word) <= stage_number()

def speak(text):
    safe = text.replace("\\", "\\\\").replace("'", "\\'")
    display(Javascript(f"""
        var msg = new SpeechSynthesisUtterance('{safe}');
        msg.rate = 0.9;
        window.speechSynthesis.cancel();
        window.speechSynthesis.speak(msg);
    """))

def word_count_boost(word):
    return PROFILE.get("word_counts", {}).get(word, 0) * 4

def recent_boost(word, lookback=20):
    history = PROFILE.get("tap_history", [])[-lookback:]
    score = 0
    for idx, tap in enumerate(reversed(history)):
        if tap.get("word") == word:
            score += max(1, lookback - idx)
    return score * 1.5

def grammar_boost(word):
    phrase = current_phrase()
    word_type = WORDS.get(word, {}).get("type", "utility")
    if phrase in ["I want", "I need"] and word == "to":
        return 40
    if phrase in ["I want to", "I need to"] and word_type == "verb":
        return 25
    if phrase.endswith("go") and word in ["potty", "bathroom", "outside", "home"]:
        return 35
    if phrase in ["I am", "I feel"] and word_type in ["feeling", "safety"]:
        return 30
    return 0

def get_predictions():
    phrase = current_phrase()
    used = set(sentence)
    current_stage = stage_number()
    limit = 9 if current_stage == 1 else 20 if current_stage == 2 else 32
    candidates = []

    strict_mode = PROFILE.get("strict_prediction_mode", True)

    def add_candidate(word, score, source=""):
        if word not in WORDS:
            return
        if not is_word_allowed_for_stage(word):
            return
        if word in used and word not in {"more", "please", "again", "thank you"}:
            return
        candidates.append((word, score, source))

    if not phrase:
        for word in STAGE_BOARDS.get(selected_stage(), []):
            add_candidate(word, WORDS.get(word, {}).get("base_weight", 40), "stage_board")
    else:
        if phrase in GRAPH:
            for word, score in GRAPH[phrase].items():
                add_candidate(word, score, "exact_phrase")

        enough = len({w for w, s, src in candidates}) >= min(limit, 6)

        if not enough:
            if sentence and sentence[-1] in GRAPH:
                for word, score in GRAPH[sentence[-1]].items():
                    add_candidate(word, score * 0.35, "last_word")

        if not strict_mode or len({w for w, s, src in candidates}) < min(limit, 4):
            fallback_pool = list(STAGE_BOARDS.get(selected_stage(), []))
            if current_stage >= 2:
                fallback_pool += STAGE_BOARDS.get("Stage 1: Emerging Communicator", [])
            if current_stage >= 3:
                fallback_pool += STAGE_BOARDS.get("Stage 2: Sentence Builder", [])
            for word in fallback_pool:
                add_candidate(word, WORDS.get(word, {}).get("base_weight", 40) * 0.18, "fallback")

    scored = []
    for word, base, source in candidates:
        source_bonus = 40 if source == "exact_phrase" else 10 if source == "last_word" else 0
        context_bonus = 0
        active_context = PROFILE.get("active_context", "Home")
        if word in PROFILE.get("context_boards", {}).get(active_context, []):
            context_bonus = 25
        status = PROFILE.get("learning_status", {}).get(word, "learning")
        learning_bonus = 15 if status == "known" else 8 if status == "learning" else 3
        scored.append((word, base + source_bonus + context_bonus + learning_bonus + word_count_boost(word) + recent_boost(word) + grammar_boost(word)))

    out = []
    seen = set()
    for word, score in sorted(scored, key=lambda x: x[1], reverse=True):
        if word in seen:
            continue
        seen.add(word)
        out.append(word)
        if len(out) >= limit:
            break
    return out

def record_tap(word):
    phrase = current_phrase()
    PROFILE.setdefault("word_counts", {})
    PROFILE.setdefault("phrase_counts", {})
    PROFILE.setdefault("tap_history", [])
    PROFILE["word_counts"][word] = PROFILE["word_counts"].get(word, 0) + 1
    PROFILE["phrase_counts"][phrase] = PROFILE["phrase_counts"].get(phrase, 0) + 1
    PROFILE["tap_history"].append({"word": word, "phrase": phrase, "sentence_length": len(sentence), "time": datetime.now().isoformat()})
    PROFILE["tap_history"] = PROFILE["tap_history"][-2000:]
    save_json(PROFILE_FILE, PROFILE)

def record_sentence():
    phrase = current_phrase()
    if not phrase:
        return
    PROFILE.setdefault("sentence_history", [])
    PROFILE["sentence_history"].append({"sentence": phrase, "length": len(sentence), "time": datetime.now().isoformat()})
    PROFILE["sentence_history"] = PROFILE["sentence_history"][-1000:]
    save_json(PROFILE_FILE, PROFILE)

def add_word(word):
    sentence.append(word)
    record_tap(word)
    speak(word)
    render_board()

def clear_sentence():
    sentence.clear()
    render_board()

def backspace():
    if sentence:
        sentence.pop()
    render_board()

def speak_sentence():
    if sentence:
        speak(current_phrase())
        record_sentence()

def speak_last_word():
    if sentence:
        speak(sentence[-1])

def change_stage(stage):
    PROFILE["selected_stage"] = stage
    sentence.clear()
    save_all()
    render_board()

def toggle_parent_tools():
    global parent_tools_open
    parent_tools_open = not parent_tools_open
    render_board()

def infer_type(word):
    lower = word.lower()
    if lower in ["mom", "dad", "teacher", "doctor", "grandma", "grandpa"]:
        return "person", "👤"
    if lower in ["pizza", "apple", "banana", "water", "juice", "milk", "snack", "food"]:
        return "food", "🍽️"
    if lower in ["happy", "sad", "mad", "scared", "hurt", "hungry", "thirsty", "sleepy", "tired", "sick"]:
        return "feeling", "😊"
    if lower in ["go", "eat", "drink", "play", "run", "walk", "jump", "see", "have", "sleep"]:
        return "verb", "🏃"
    if lower in ["home", "school", "outside", "bathroom", "potty", "bed", "park", "car"]:
        return "place", "🏠"
    if lower in ["help", "stop", "danger", "emergency"]:
        return "safety", "🆘"
    return "object", "🔤"

def connect_word_after_phrase(word, phrase, weight=115):
    if phrase not in GRAPH:
        GRAPH[phrase] = {}
    GRAPH[phrase][word] = weight

def add_custom_word(word, word_type, image, show_after_phrase):
    global system_message
    clean = word.strip()
    phrase = show_after_phrase.strip()
    if not clean:
        system_message = "No word entered."
        render_board()
        return

    inferred_type, inferred_image = infer_type(clean)
    duplicate = clean in WORDS
    if duplicate:
        WORDS[clean]["type"] = word_type or WORDS[clean].get("type", inferred_type)
        WORDS[clean]["image"] = image.strip() or WORDS[clean].get("image", inferred_image)
        WORDS[clean]["base_weight"] = max(WORDS[clean].get("base_weight", 75), 85)
        WORDS[clean]["stage"] = min(WORDS[clean].get("stage", stage_number()), stage_number())
        system_message = f"'{clean}' already existed. Updated it instead of duplicating."
    else:
        WORDS[clean] = {"image": image.strip() or inferred_image, "type": word_type or inferred_type, "base_weight": 85, "stage": stage_number()}
        system_message = f"Added '{clean}'."

    if clean not in STAGE_BOARDS[selected_stage()]:
        STAGE_BOARDS[selected_stage()].append(clean)
    if phrase:
        connect_word_after_phrase(clean, phrase)
        system_message += f" Connected it after '{phrase}'."

    save_all()
    render_board()

def delete_word(word):
    global last_deleted_word, last_deleted_payload, system_message
    clean = word.strip()
    if not clean or clean not in WORDS:
        system_message = "Word not found."
        render_board()
        return
    last_deleted_word = clean
    last_deleted_payload = {"word_data": WORDS.get(clean), "stage_boards": {stage: words[:] for stage, words in STAGE_BOARDS.items() if clean in words}, "graph_refs": {}}
    for phrase, next_words in GRAPH.items():
        if clean in next_words:
            last_deleted_payload["graph_refs"][phrase] = next_words[clean]
    del WORDS[clean]
    for stage, words in STAGE_BOARDS.items():
        STAGE_BOARDS[stage] = [w for w in words if w != clean]
    for phrase in list(GRAPH.keys()):
        if clean in GRAPH[phrase]:
            del GRAPH[phrase][clean]
    system_message = f"Deleted '{clean}'. You can undo this."
    save_all()
    render_board()

def undo_delete_word():
    global last_deleted_word, last_deleted_payload, system_message
    if not last_deleted_word or not last_deleted_payload:
        system_message = "Nothing to undo."
        render_board()
        return
    WORDS[last_deleted_word] = last_deleted_payload["word_data"]
    for stage, words in last_deleted_payload["stage_boards"].items():
        if stage in STAGE_BOARDS and last_deleted_word not in STAGE_BOARDS[stage]:
            STAGE_BOARDS[stage].append(last_deleted_word)
    for phrase, weight in last_deleted_payload["graph_refs"].items():
        if phrase not in GRAPH:
            GRAPH[phrase] = {}
        GRAPH[phrase][last_deleted_word] = weight
    system_message = f"Restored '{last_deleted_word}'."
    last_deleted_word = None
    last_deleted_payload = None
    save_all()
    render_board()


def add_austin_baseline_1_8():
    baseline_words = {
        "apple juice": {"image": "🧃", "type": "food", "base_weight": 120, "stage": 1},
        "water": {"image": "💧", "type": "food", "base_weight": 120, "stage": 1},
        "I": {"image": "👤", "type": "pronoun", "base_weight": 130, "stage": 1},
        "want": {"image": "🤲", "type": "verb", "base_weight": 130, "stage": 1},
        "yes": {"image": "👍", "type": "utility", "base_weight": 120, "stage": 1},
        "no": {"image": "👎", "type": "utility", "base_weight": 120, "stage": 1},
        "please": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "thank you": {"image": "🙏", "type": "social", "base_weight": 115, "stage": 1},
        "and": {"image": "➕", "type": "utility", "base_weight": 75, "stage": 2},
        "awesome": {"image": "🤩", "type": "feeling", "base_weight": 95, "stage": 1},
        "good": {"image": "👍", "type": "feeling", "base_weight": 90, "stage": 1},
        "okay": {"image": "👌", "type": "feeling", "base_weight": 100, "stage": 1},
        "time": {"image": "⏰", "type": "time", "base_weight": 85, "stage": 2},
        "for": {"image": "➡️", "type": "utility", "base_weight": 82, "stage": 2},
        "bed": {"image": "🛏️", "type": "place", "base_weight": 100, "stage": 1},
        "gummies": {"image": "🍬", "type": "food", "base_weight": 110, "stage": 1},
        "french fries": {"image": "🍟", "type": "food", "base_weight": 110, "stage": 1},
        "chicken nuggets": {"image": "🍗", "type": "food", "base_weight": 115, "stage": 1},
        "I'm hungry": {"image": "🍽️", "type": "feeling", "base_weight": 115, "stage": 1},
        "I'm thirsty": {"image": "🥤", "type": "feeling", "base_weight": 115, "stage": 1},
        "hungry": {"image": "🍽️", "type": "feeling", "base_weight": 115, "stage": 1},
        "thirsty": {"image": "🥤", "type": "feeling", "base_weight": 115, "stage": 1},
    }

    for word, data in baseline_words.items():
        WORDS[word] = data

    STAGE_BOARDS["Stage 1: Emerging Communicator"] = [
        "I", "want", "yes",
        "no", "please", "water",
        "apple juice", "chicken nuggets", "potty"
    ]

    STAGE_BOARDS["Stage 2: Sentence Builder"] = [
        "I", "want", "please", "yes",
        "no", "water", "apple juice", "milk",
        "gummies", "french fries", "chicken nuggets", "food",
        "I'm hungry", "I'm thirsty", "hungry", "thirsty",
        "time", "for", "bed", "okay",
        "good", "awesome", "and", "potty",
        "go", "eat", "drink", "play",
        "mom", "dad", "help", "all done"
    ]

    baseline_graph = {
        "I": {"want": 150, "am": 90, "need": 85, "feel": 70},
        "I want": {
            "apple juice": 140, "water": 135, "chicken nuggets": 130,
            "french fries": 125, "gummies": 120, "milk": 115,
            "please": 100, "food": 95, "to": 90
        },
        "I want apple juice": {"please": 130, "and": 75, "more": 70},
        "I want water": {"please": 130, "and": 75, "more": 70},
        "I want chicken nuggets": {"please": 130, "and": 80, "more": 70},
        "I want french fries": {"please": 130, "and": 80, "more": 70},
        "I want gummies": {"please": 130, "and": 75, "more": 70},
        "I want milk": {"please": 130, "and": 75, "more": 70},
        "time": {"for": 140, "bed": 80},
        "time for": {"bed": 150, "potty": 80, "bathroom": 75},
        "I'm": {"hungry": 140, "thirsty": 135},
        "I am": {"hungry": 130, "thirsty": 125, "okay": 120, "good": 110, "awesome": 105, "happy": 90, "sad": 80, "mad": 78, "hurt": 85},
        "I feel": {"hungry": 130, "thirsty": 125, "okay": 120, "good": 110, "awesome": 105, "happy": 90, "sad": 80, "mad": 78, "hurt": 85},
        "I'm hungry": {"food": 130, "chicken nuggets": 120, "french fries": 110, "please": 100},
        "I'm thirsty": {"water": 130, "apple juice": 125, "milk": 110, "please": 100},
        "hungry": {"food": 130, "chicken nuggets": 120, "french fries": 110, "please": 90},
        "thirsty": {"water": 130, "apple juice": 125, "milk": 110, "please": 90},
        "yes": {"please": 100, "awesome": 80, "good": 75},
        "no": {"thank you": 100, "stop": 80, "all done": 75},
        "please": {"water": 115, "apple juice": 110, "chicken nuggets": 105, "gummies": 100},
        "and": {"water": 100, "apple juice": 100, "chicken nuggets": 95, "french fries": 95, "gummies": 90},
    }

    for phrase, words in baseline_graph.items():
        GRAPH.setdefault(phrase, {})
        GRAPH[phrase].update(words)

    PROFILE["austin_baseline_loaded"] = True
    save_all()

def classify_word_for_tree(word):
    lower = word.lower().strip()
    if lower.startswith("i'm ") or lower.startswith("im "):
        return "phrase_state"
    if any(token in lower for token in ["juice", "water", "milk", "gummies", "fries", "nuggets", "food", "apple", "banana", "pizza", "snack", "cracker", "cereal", "yogurt", "drink", "thirsty", "hungry"]):
        return "food"
    if lower in ["happy", "sad", "mad", "okay", "good", "awesome", "hurt", "sick", "tired", "sleepy", "scared"]:
        return "feeling"
    if lower in ["bed", "potty", "bathroom", "home", "school", "outside", "park", "daycare"]:
        return "place"
    if lower in ["go", "eat", "drink", "play", "jump", "run", "walk", "sleep", "watch", "read", "open", "close"]:
        return "verb"
    if lower in ["mom", "dad", "grandma", "grandpa", "teacher"]:
        return "person"
    if lower in ["yes", "no", "please", "thank you", "thankyou", "and", "for", "to", "more", "all done"]:
        return "utility"
    return "object"

def auto_icon_for_word(word, kind):
    lower = word.lower().strip()
    if "juice" in lower: return "🧃"
    if "water" in lower: return "💧"
    if "milk" in lower: return "🥛"
    if "gummies" in lower: return "🍬"
    if "fries" in lower: return "🍟"
    if "nuggets" in lower: return "🍗"
    if "hungry" in lower or "food" in lower: return "🍽️"
    if "thirsty" in lower or "drink" in lower: return "🥤"
    if "bed" in lower: return "🛏️"
    if "potty" in lower or "bathroom" in lower: return "🚽"
    if "awesome" in lower: return "🤩"
    if "good" in lower: return "👍"
    if "okay" in lower: return "👌"
    if "please" in lower: return "🙏"
    if "thank" in lower: return "🙏"
    if kind == "food": return "🍽️"
    if kind == "feeling": return "😊"
    if kind == "place": return "🏠"
    if kind == "verb": return "🏃"
    if kind == "person": return "👤"
    return "🔤"

def auto_tree_words(raw_text):
    global system_message

    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
    added = []
    linked = 0

    for raw in lines:
        normalized = raw.strip().replace("thankyou", "thank you").replace("Im ", "I'm ").replace("i'm ", "I'm ")
        if not normalized:
            continue

        kind = classify_word_for_tree(normalized)
        word_type = "feeling" if kind == "phrase_state" else kind
        if word_type not in TYPE_STYLES:
            word_type = "object"

        stage = 1 if normalized in [
            "apple juice", "water", "I", "want", "yes", "no", "please", "thank you",
            "awesome", "good", "okay", "bed", "gummies", "french fries",
            "chicken nuggets", "I'm hungry", "I'm thirsty"
        ] else 2

        WORDS[normalized] = {
            "image": auto_icon_for_word(normalized, kind),
            "type": word_type,
            "base_weight": 115 if stage == 1 else 85,
            "stage": stage
        }

        if normalized not in STAGE_BOARDS["Stage 2: Sentence Builder"]:
            STAGE_BOARDS["Stage 2: Sentence Builder"].append(normalized)

        if kind == "food":
            for phrase in ["I want", "I want to drink", "I want to eat", "I'm hungry", "I'm thirsty", "and"]:
                GRAPH.setdefault(phrase, {})[normalized] = 115
                linked += 1
        elif kind == "phrase_state":
            GRAPH.setdefault("I", {})[normalized] = 115
            if "hungry" in normalized.lower():
                GRAPH.setdefault(normalized, {})["food"] = 120
                GRAPH.setdefault(normalized, {})["chicken nuggets"] = 110
                GRAPH.setdefault(normalized, {})["french fries"] = 105
            if "thirsty" in normalized.lower():
                GRAPH.setdefault(normalized, {})["water"] = 120
                GRAPH.setdefault(normalized, {})["apple juice"] = 115
                GRAPH.setdefault(normalized, {})["milk"] = 105
            linked += 4
        elif kind == "feeling":
            GRAPH.setdefault("I am", {})[normalized] = 115
            GRAPH.setdefault("I feel", {})[normalized] = 110
            GRAPH.setdefault("yes", {})[normalized] = 80
            linked += 3
        elif kind == "place":
            GRAPH.setdefault("time for", {})[normalized] = 120
            GRAPH.setdefault("I want to go", {})[normalized] = 105
            GRAPH.setdefault("I need to go", {})[normalized] = 105
            linked += 3
        elif kind == "verb":
            GRAPH.setdefault("I want to", {})[normalized] = 105
            GRAPH.setdefault("I need to", {})[normalized] = 100
            GRAPH.setdefault("can I", {})[normalized] = 95
            linked += 3
        elif kind == "utility":
            if normalized == "for":
                GRAPH.setdefault("time", {})["for"] = 140
            elif normalized == "and":
                GRAPH.setdefault("I want apple juice", {})["and"] = 80
                GRAPH.setdefault("I want water", {})["and"] = 80
                GRAPH.setdefault("I want chicken nuggets", {})["and"] = 80
            elif normalized == "please":
                GRAPH.setdefault("I want", {})["please"] = 100
            elif normalized == "thank you":
                GRAPH.setdefault("no", {})["thank you"] = 100
            linked += 2
        else:
            GRAPH.setdefault("I want", {})[normalized] = 85
            GRAPH.setdefault("I like", {})[normalized] = 80
            linked += 2

        added.append(normalized)

    save_all()
    system_message = f"Auto-tree complete. Added/updated {len(added)} words and created about {linked} prediction links."
    render_board()

def vocabulary_growth_pack_for_austin():
    return [
        "more apple juice",
        "more water",
        "I want more",
        "I want gummies",
        "I want chicken nuggets",
        "I want french fries",
        "I am hungry",
        "I am thirsty",
        "time for bed",
        "bedtime",
        "all done",
        "help please",
        "no thank you",
        "yes please",
        "good job",
        "I feel good",
        "I feel okay",
        "I need help",
        "I need water",
        "I need potty"
    ]

add_austin_baseline_1_8()



def add_emotional_social_layer_1_9():
    social_emotion_words = {
        "you": {"image": "👉", "type": "pronoun", "base_weight": 95, "stage": 1},
        "love": {"image": "💗", "type": "social", "base_weight": 110, "stage": 1},
        "I love you": {"image": "💗", "type": "social", "base_weight": 130, "stage": 1},
        "thank you": {"image": "🙏", "type": "social", "base_weight": 120, "stage": 1},
        "no thank you": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "yes please": {"image": "🙏", "type": "social", "base_weight": 105, "stage": 1},
        "help please": {"image": "🆘", "type": "safety", "base_weight": 110, "stage": 1},
        "good job": {"image": "🌟", "type": "social", "base_weight": 85, "stage": 2},

        "happy": {"image": "😊", "type": "feeling", "base_weight": 100, "stage": 1},
        "sad": {"image": "😢", "type": "feeling", "base_weight": 95, "stage": 1},
        "mad": {"image": "😡", "type": "feeling", "base_weight": 95, "stage": 1},
        "scared": {"image": "😨", "type": "feeling", "base_weight": 90, "stage": 1},
        "hurt": {"image": "🤕", "type": "safety", "base_weight": 100, "stage": 1},
        "hungry": {"image": "🍽️", "type": "feeling", "base_weight": 115, "stage": 1},
        "thirsty": {"image": "🥤", "type": "feeling", "base_weight": 115, "stage": 1},
        "sleepy": {"image": "😴", "type": "feeling", "base_weight": 95, "stage": 1},
        "tired": {"image": "🥱", "type": "feeling", "base_weight": 90, "stage": 1},
        "sick": {"image": "🤢", "type": "feeling", "base_weight": 90, "stage": 1},
        "excited": {"image": "🤩", "type": "feeling", "base_weight": 82, "stage": 2},
        "calm": {"image": "😌", "type": "feeling", "base_weight": 78, "stage": 2},
        "frustrated": {"image": "😤", "type": "feeling", "base_weight": 72, "stage": 2},
    }

    for word, data in social_emotion_words.items():
        WORDS[word] = data

    # Keep Stage 1 Austin-simple but include direct social/emotional access.
    STAGE_BOARDS["Stage 1: Emerging Communicator"] = [
        "I", "want", "yes",
        "no", "please", "water",
        "apple juice", "chicken nuggets", "I love you"
    ]

    # Stage 2 includes the emotional/social words clearly.
    stage2_add = [
        "you", "love", "I love you", "thank you", "no thank you", "yes please",
        "help please", "happy", "sad", "mad", "scared", "hurt",
        "hungry", "thirsty", "sleepy", "tired", "sick", "excited", "calm"
    ]
    for w in stage2_add:
        if w not in STAGE_BOARDS["Stage 2: Sentence Builder"]:
            STAGE_BOARDS["Stage 2: Sentence Builder"].append(w)

    emotional_graph = {
        "I": {
            "love": 150,
            "want": 145,
            "need": 130,
            "am": 125,
            "feel": 120,
            "like": 90
        },
        "I love": {
            "you": 160,
            "mom": 120,
            "dad": 120,
            "jumpy": 75,
            "play": 70
        },
        "I love you": {
            "thank you": 90,
            "mom": 80,
            "dad": 80
        },
        "I am": {
            "happy": 130,
            "sad": 120,
            "mad": 118,
            "scared": 115,
            "hurt": 125,
            "hungry": 130,
            "thirsty": 128,
            "sleepy": 115,
            "tired": 110,
            "sick": 105,
            "okay": 125,
            "good": 115,
            "awesome": 110,
            "excited": 90,
            "calm": 85,
            "frustrated": 80
        },
        "I feel": {
            "happy": 125,
            "sad": 120,
            "mad": 118,
            "scared": 115,
            "hurt": 125,
            "hungry": 125,
            "thirsty": 123,
            "sleepy": 115,
            "tired": 110,
            "sick": 105,
            "okay": 120,
            "good": 110,
            "awesome": 105,
            "excited": 90,
            "calm": 85,
            "frustrated": 80
        },
        "yes": {
            "please": 120,
            "yes please": 130,
            "awesome": 95,
            "good": 90,
            "I love you": 75
        },
        "no": {
            "thank you": 120,
            "no thank you": 130,
            "stop": 95,
            "all done": 90
        },
        "help": {
            "please": 125,
            "help please": 140,
            "me": 120,
            "mom": 100,
            "dad": 100
        },
        "thank you": {
            "I love you": 90,
            "mom": 70,
            "dad": 70
        }
    }

    for phrase, next_words in emotional_graph.items():
        GRAPH.setdefault(phrase, {})
        GRAPH[phrase].update(next_words)

    save_all()

add_emotional_social_layer_1_9()



def add_phrase_chain(phrase, stage=1, image="💬", word_type="social", weight=130):
    # Adds a full phrase as a direct button AND adds its word-by-word path.
    global system_message
    clean = phrase.strip()
    if not clean:
        return

    WORDS[clean] = {"image": image, "type": word_type, "base_weight": weight, "stage": stage}
    if clean not in STAGE_BOARDS[selected_stage()]:
        STAGE_BOARDS[selected_stage()].append(clean)

    parts = clean.split()
    if len(parts) >= 2:
        # Example: I love you
        # GRAPH["I"] -> love
        # GRAPH["I love"] -> you
        for i in range(1, len(parts)):
            prefix = " ".join(parts[:i])
            next_word = parts[i]
            GRAPH.setdefault(prefix, {})[next_word] = weight + max(0, 20 - i * 5)

            if next_word not in WORDS:
                inferred_type, inferred_image = infer_type(next_word)
                WORDS[next_word] = {"image": inferred_image, "type": inferred_type, "base_weight": 80, "stage": stage}

        # Also make the direct phrase available after the first word.
        GRAPH.setdefault(parts[0], {})[clean] = weight

    save_all()
    system_message = f"Added phrase chain: {clean}"
    render_board()

def run_engine_self_tests():
    results = []

    # stage_number exists and returns int
    try:
        sn = stage_number()
        results.append(("stage_number defined", isinstance(sn, int)))
    except Exception:
        results.append(("stage_number defined", False))

    # Core social phrase exists
    results.append(("I love you exists", "I love you" in WORDS))
    results.append(("I -> love path", "love" in GRAPH.get("I", {})))
    results.append(("I love -> you path", "you" in GRAPH.get("I love", {})))

    # Austin baseline exists
    for word in ["apple juice", "water", "chicken nuggets", "gummies", "I'm hungry", "I'm thirsty"]:
        results.append((f"{word} exists", word in WORDS))

    # Stage 1 should not leak stage 2/3 words on stage board
    leaks = validate_stage_leaks()
    results.append(("no stage board leaks", not bool(leaks)))
    results.append(("concept network exists", bool(PROFILE.get("concept_network"))))
    results.append(("relationship graph exists", bool(PROFILE.get("relationship_graph"))))
    results.append(("context boards exist", bool(PROFILE.get("context_boards"))))
    results.append(("developmental maps exist", bool(PROFILE.get("developmental_maps"))))
    results.append(("learning status exists", bool(PROFILE.get("learning_status"))))
    results.append(("knowledge objects exist", bool(PROFILE.get("knowledge_objects"))))
    results.append(("knowledge object validation clean", len(validate_knowledge_objects()) == 0))

    passed = sum(1 for name, ok in results if ok)
    total = len(results)

    html = f"<h3>Self-Test Results: {passed}/{total} passed</h3>"
    html += "<ul>"
    for name, ok in results:
        mark = "✅" if ok else "❌"
        html += f"<li>{mark} {name}</li>"
    html += "</ul>"

    return html



# ============================================================
# TalkFree Language Genome 1.11
# ============================================================

PROFILE.setdefault("active_context", "Home")
PROFILE.setdefault("parent_goals", ["Expand requesting", "Build emotions", "Grow food/drink vocabulary"])
PROFILE.setdefault("learning_status", {})
PROFILE.setdefault("concept_network", {})
PROFILE.setdefault("relationship_graph", {})
PROFILE.setdefault("context_boards", {})
PROFILE.setdefault("developmental_maps", {})

CONCEPT_NETWORK = {
    "drinks": ["water", "milk", "apple juice", "juice", "orange juice", "chocolate milk", "cup", "bottle", "straw"],
    "foods": ["food", "snack", "gummies", "french fries", "chicken nuggets", "pizza", "apple", "banana", "yogurt", "cracker", "cereal", "mac and cheese"],
    "feelings": ["happy", "sad", "mad", "scared", "hurt", "hungry", "thirsty", "sleepy", "tired", "sick", "okay", "good", "awesome", "excited", "calm", "frustrated"],
    "people": ["mom", "dad", "you", "me", "grandma", "grandpa", "teacher", "doctor", "friend"],
    "places": ["home", "bed", "potty", "bathroom", "outside", "school", "park", "playground", "daycare", "car", "kitchen", "bedroom"],
    "actions": ["go", "eat", "drink", "play", "jump", "run", "walk", "sleep", "help", "open", "close", "wash", "brush", "read", "watch"],
    "social": ["I love you", "please", "thank you", "yes please", "no thank you", "help please", "good job", "hello", "bye"],
    "body": ["belly", "head", "ear", "teeth", "hand", "foot", "mouth", "eyes", "nose"],
    "bedtime": ["time", "for", "bed", "sleep", "blanket", "quiet", "tired", "sleepy", "good night"],
    "bathroom": ["potty", "bathroom", "pee", "poop", "wipe", "wash", "hands", "help"],
    "play": ["play", "jump", "jumpy", "trampoline", "toy", "tablet", "tv", "music", "outside"]
}

SYNONYM_NETWORK = {
    "happy": ["good", "awesome", "excited"],
    "sad": ["upset", "cry"],
    "mad": ["angry", "frustrated"],
    "thirsty": ["drink", "water", "juice"],
    "hungry": ["eat", "food", "snack"],
    "bed": ["sleep", "tired", "sleepy", "good night"],
    "potty": ["bathroom", "pee", "poop"],
    "apple juice": ["juice", "drink"],
    "chicken nuggets": ["food", "eat"],
    "french fries": ["food", "eat"],
    "I love you": ["love", "hug"]
}

DEVELOPMENTAL_MAPS = {
    "requesting": {"stage": 1, "starters": ["I want", "more", "please", "help"], "targets": ["food", "drink", "play", "potty", "hug"]},
    "answering": {"stage": 1, "starters": ["yes", "no", "okay"], "targets": ["yes please", "no thank you", "all done"]},
    "emotion_labeling": {"stage": 1, "starters": ["I am", "I feel", "I'm"], "targets": ["happy", "sad", "mad", "hurt", "hungry", "thirsty", "sleepy"]},
    "social_connection": {"stage": 1, "starters": ["I love", "thank you", "please"], "targets": ["you", "mom", "dad", "I love you"]},
    "commenting": {"stage": 2, "starters": ["it is", "that is", "I like"], "targets": ["good", "awesome", "fun", "loud", "quiet"]},
    "question_asking": {"stage": 3, "starters": ["what", "where", "who", "why", "when", "how"], "targets": ["mom", "dad", "toy", "potty", "go"]}
}

CONTEXT_BOARDS = {
    "Home": ["I", "want", "water", "apple juice", "chicken nuggets", "I love you", "potty", "help", "all done"],
    "Food": ["I", "want", "water", "apple juice", "milk", "gummies", "french fries", "chicken nuggets", "I'm hungry"],
    "Bedtime": ["time", "for", "bed", "sleepy", "tired", "I love you", "water", "potty", "blanket"],
    "Bathroom": ["I", "need", "potty", "bathroom", "help", "all done", "wash", "hurt", "yes"],
    "Play": ["I", "want", "play", "jump", "jumpy", "trampoline", "again", "more", "all done"],
    "Feelings": ["I", "am", "feel", "happy", "sad", "mad", "hurt", "okay", "I love you"],
    "Emergency": ["help", "hurt", "sick", "scared", "mom", "dad", "doctor", "stop", "potty"]
}

RELATIONSHIP_GRAPH = {
    "water": {"category": "drink", "verbs": ["drink", "want", "need"], "related": ["apple juice", "milk", "cup"], "contexts": ["Home", "Food", "Bedtime"]},
    "apple juice": {"category": "drink", "verbs": ["drink", "want"], "related": ["water", "milk", "juice"], "contexts": ["Home", "Food"]},
    "chicken nuggets": {"category": "food", "verbs": ["eat", "want"], "related": ["french fries", "food", "snack"], "contexts": ["Home", "Food"]},
    "french fries": {"category": "food", "verbs": ["eat", "want"], "related": ["chicken nuggets", "food"], "contexts": ["Food"]},
    "gummies": {"category": "food", "verbs": ["eat", "want"], "related": ["snack", "please"], "contexts": ["Food", "Home"]},
    "bed": {"category": "place", "verbs": ["go", "sleep"], "related": ["sleepy", "tired", "blanket"], "contexts": ["Bedtime"]},
    "potty": {"category": "bathroom", "verbs": ["go", "need"], "related": ["bathroom", "help", "all done"], "contexts": ["Bathroom", "Home"]},
    "I love you": {"category": "social", "verbs": ["love"], "related": ["mom", "dad", "hug", "thank you"], "contexts": ["Home", "Bedtime", "Feelings"]},
    "happy": {"category": "feeling", "verbs": ["am", "feel"], "related": ["good", "awesome", "excited"], "contexts": ["Feelings"]},
    "hurt": {"category": "safety", "verbs": ["am", "feel", "need"], "related": ["help", "mom", "dad", "doctor"], "contexts": ["Emergency", "Feelings"]}
}

def apply_language_genome_1_11():
    for concept, words in CONCEPT_NETWORK.items():
        for word in words:
            if word not in WORDS:
                inferred_type, inferred_image = infer_type(word)
                stage = 1 if word in ["water", "milk", "apple juice", "food", "happy", "sad", "mad", "hurt", "I love you", "please", "thank you"] else 2
                WORDS[word] = {"image": inferred_image, "type": inferred_type, "base_weight": 65, "stage": stage}
            WORDS[word]["concept"] = concept
            WORDS[word]["asset"] = safe_asset_path(word, WORDS[word].get("type", "object"))

    for drink in CONCEPT_NETWORK["drinks"]:
        if drink in WORDS:
            GRAPH.setdefault("I want", {})[drink] = max(GRAPH.get("I want", {}).get(drink, 0), 90)
            GRAPH.setdefault("I want to drink", {})[drink] = max(GRAPH.get("I want to drink", {}).get(drink, 0), 100)
            GRAPH.setdefault("I'm thirsty", {})[drink] = max(GRAPH.get("I'm thirsty", {}).get(drink, 0), 100)

    for food in CONCEPT_NETWORK["foods"]:
        if food in WORDS:
            GRAPH.setdefault("I want", {})[food] = max(GRAPH.get("I want", {}).get(food, 0), 90)
            GRAPH.setdefault("I want to eat", {})[food] = max(GRAPH.get("I want to eat", {}).get(food, 0), 100)
            GRAPH.setdefault("I'm hungry", {})[food] = max(GRAPH.get("I'm hungry", {}).get(food, 0), 100)

    for feeling in CONCEPT_NETWORK["feelings"]:
        if feeling in WORDS:
            GRAPH.setdefault("I am", {})[feeling] = max(GRAPH.get("I am", {}).get(feeling, 0), 95)
            GRAPH.setdefault("I feel", {})[feeling] = max(GRAPH.get("I feel", {}).get(feeling, 0), 95)

    PROFILE["concept_network"] = CONCEPT_NETWORK
    PROFILE["relationship_graph"] = RELATIONSHIP_GRAPH
    PROFILE["context_boards"] = CONTEXT_BOARDS
    PROFILE["developmental_maps"] = DEVELOPMENTAL_MAPS
    PROFILE["synonym_network"] = SYNONYM_NETWORK

    for word in WORDS:
        PROFILE["learning_status"].setdefault(word, "learning")
    for word in ["I", "want", "yes", "no", "water", "apple juice", "chicken nuggets", "please"]:
        PROFILE["learning_status"][word] = "known"
    for word in ["I love you", "I'm hungry", "I'm thirsty", "potty", "gummies", "french fries"]:
        PROFILE["learning_status"][word] = "learning"

    mine_phrases_from_concepts()
    save_all()

def mine_phrases_from_concepts():
    for word, rel in RELATIONSHIP_GRAPH.items():
        if word not in WORDS:
            continue
        category = rel.get("category", "")
        verbs = rel.get("verbs", [])
        related = rel.get("related", [])

        if "want" in verbs:
            GRAPH.setdefault("I want", {})[word] = max(GRAPH.get("I want", {}).get(word, 0), 100)
            GRAPH.setdefault("I want more", {})[word] = 90
            GRAPH.setdefault("I want " + word, {})["please"] = 100
        if "need" in verbs:
            GRAPH.setdefault("I need", {})[word] = max(GRAPH.get("I need", {}).get(word, 0), 95)
        if "drink" in verbs:
            GRAPH.setdefault("I want to drink", {})[word] = max(GRAPH.get("I want to drink", {}).get(word, 0), 105)
        if "eat" in verbs:
            GRAPH.setdefault("I want to eat", {})[word] = max(GRAPH.get("I want to eat", {}).get(word, 0), 105)
        if category == "feeling":
            GRAPH.setdefault("I am", {})[word] = max(GRAPH.get("I am", {}).get(word, 0), 100)
            GRAPH.setdefault("I feel", {})[word] = max(GRAPH.get("I feel", {}).get(word, 0), 100)
        for r in related:
            if r in WORDS:
                GRAPH.setdefault(word, {})[r] = 70

def change_context(context_name):
    PROFILE["active_context"] = context_name
    if context_name in CONTEXT_BOARDS:
        current = selected_stage()
        merged = list(dict.fromkeys(CONTEXT_BOARDS[context_name] + STAGE_BOARDS.get(current, [])))
        STAGE_BOARDS[current] = [w for w in merged if w in WORDS and is_word_allowed_for_stage(w)]
    save_all()
    render_board()

def get_goal_suggestions():
    goals = PROFILE.get("parent_goals", [])
    suggestions = []
    if "Expand requesting" in goals:
        suggestions += ["I want more", "I want help", "I want play", "I want drink"]
    if "Build emotions" in goals:
        suggestions += ["I feel okay", "I feel hurt", "I am happy", "I am mad", "I need a break"]
    if "Grow food/drink vocabulary" in goals:
        suggestions += ["cup", "bottle", "straw", "orange juice", "snack", "cracker", "yogurt"]
    if "Bathroom independence" in goals:
        suggestions += ["I need potty", "I need help", "all done", "wash hands"]
    if "Bedtime routine" in goals:
        suggestions += ["time for bed", "I am sleepy", "I need blanket", "good night"]
    return list(dict.fromkeys(suggestions))

def add_goal_suggestions():
    global system_message
    suggestions = get_goal_suggestions()
    auto_tree_words("\\n".join(suggestions))
    system_message = "Added " + str(len(suggestions)) + " goal-based suggestions."
    render_board()

def vocabulary_score_summary():
    counts = {"known": 0, "learning": 0, "suggested": 0, "mastered": 0, "rarely_used": 0}
    for word, status in PROFILE.get("learning_status", {}).items():
        counts[status] = counts.get(status, 0) + 1
    return counts

def emotion_coaching_paths():
    coaching = {
        "sad": ["hug", "help", "mom", "dad"],
        "mad": ["break", "quiet", "help", "all done"],
        "scared": ["hug", "mom", "dad", "help"],
        "hurt": ["help", "mom", "dad", "doctor"],
        "hungry": ["food", "snack", "chicken nuggets", "french fries"],
        "thirsty": ["water", "apple juice", "milk"],
        "sleepy": ["bed", "sleep", "blanket"],
        "sick": ["help", "doctor", "mom", "dad"]
    }
    for emotion, needs in coaching.items():
        GRAPH.setdefault(emotion, {})
        GRAPH.setdefault("I feel " + emotion, {})
        GRAPH.setdefault("I am " + emotion, {})
        for need in needs:
            if need in WORDS:
                GRAPH[emotion][need] = 100
                GRAPH["I feel " + emotion][need] = 100
                GRAPH["I am " + emotion][need] = 100
    save_all()

def genome_status_html():
    vocab_scores = vocabulary_score_summary()
    html = "<h3>Language Genome Status</h3><ul>"
    html += "<li>Concept categories: " + str(len(CONCEPT_NETWORK)) + "</li>"
    html += "<li>Relationship graph objects: " + str(len(RELATIONSHIP_GRAPH)) + "</li>"
    html += "<li>Context modes: " + str(len(CONTEXT_BOARDS)) + "</li>"
    html += "<li>Developmental maps: " + str(len(DEVELOPMENTAL_MAPS)) + "</li>"
    html += "<li>Synonym groups: " + str(len(SYNONYM_NETWORK)) + "</li>"
    html += "</ul><h4>Vocabulary Status</h4><ul>"
    for k, v in vocab_scores.items():
        html += "<li>" + str(k) + ": " + str(v) + "</li>"
    html += "</ul>"
    return html

emotion_coaching_paths()
apply_language_genome_1_11()



# ============================================================
# Parent Insight Layer 1.12
# ============================================================

PROFILE.setdefault("insight_enabled", True)
PROFILE.setdefault("insight_notes", [])

def count_by_word_type():
    counts = {}
    for word, count in PROFILE.get("word_counts", {}).items():
        word_type = WORDS.get(word, {}).get("type", "unknown")
        counts[word_type] = counts.get(word_type, 0) + count
    return counts

def count_by_concept():
    counts = {}
    for word, count in PROFILE.get("word_counts", {}).items():
        concept = WORDS.get(word, {}).get("concept", WORDS.get(word, {}).get("type", "unknown"))
        counts[concept] = counts.get(concept, 0) + count
    return counts

def top_n_from_dict(data, n=10):
    return sorted(data.items(), key=lambda x: x[1], reverse=True)[:n]

def phrase_length_stats():
    sentences = PROFILE.get("sentence_history", [])
    if not sentences:
        return {"completed_sentences": 0, "average_length": 0, "max_length": 0}
    lengths = [s.get("length", 0) for s in sentences]
    avg = round(sum(lengths) / len(lengths), 2) if lengths else 0
    return {"completed_sentences": len(sentences), "average_length": avg, "max_length": max(lengths)}

def communication_functions():
    word_counts = PROFILE.get("word_counts", {})
    phrase_counts = PROFILE.get("phrase_counts", {})
    functions = {
        "requesting": 0,
        "emotion": 0,
        "social": 0,
        "safety": 0,
        "food_drink": 0,
        "bathroom": 0,
        "bedtime": 0
    }

    for word, count in word_counts.items():
        lower = word.lower()
        word_type = WORDS.get(word, {}).get("type", "")
        concept = WORDS.get(word, {}).get("concept", "")

        if lower in ["want", "more", "please"] or lower.startswith("i want"):
            functions["requesting"] += count
        if word_type == "feeling" or concept == "feelings" or lower in ["happy", "sad", "mad", "hurt", "hungry", "thirsty", "okay", "good", "awesome"]:
            functions["emotion"] += count
        if word_type == "social" or lower in ["i love you", "thank you", "yes please", "no thank you"]:
            functions["social"] += count
        if word_type == "safety" or lower in ["help", "hurt", "sick", "stop"]:
            functions["safety"] += count
        if concept in ["foods", "drinks"] or lower in ["water", "apple juice", "milk", "chicken nuggets", "french fries", "gummies"]:
            functions["food_drink"] += count
        if lower in ["potty", "bathroom", "pee", "poop"]:
            functions["bathroom"] += count
        if lower in ["bed", "sleep", "sleepy", "tired", "time"]:
            functions["bedtime"] += count

    for phrase, count in phrase_counts.items():
        lower = phrase.lower()
        if "i want" in lower:
            functions["requesting"] += count
        if "i feel" in lower or "i am" in lower or "i'm hungry" in lower or "i'm thirsty" in lower:
            functions["emotion"] += count
        if "love" in lower or "thank" in lower:
            functions["social"] += count
        if "potty" in lower or "bathroom" in lower:
            functions["bathroom"] += count
        if "bed" in lower:
            functions["bedtime"] += count

    return functions

def identify_strengths_and_gaps():
    functions = communication_functions()
    strengths = []
    gaps = []

    for name, count in functions.items():
        if count >= 5:
            strengths.append(name)
        elif count <= 1:
            gaps.append(name)

    return strengths, gaps

def next_best_words():
    strengths, gaps = identify_strengths_and_gaps()
    suggestions = []

    # Always grow from Austin baseline first.
    if "food_drink" in strengths or not strengths:
        suggestions += ["cup", "straw", "orange juice", "cracker", "yogurt", "more water", "more apple juice"]

    if "emotion" in gaps:
        suggestions += ["I feel okay", "I feel hurt", "I am happy", "I am mad", "I need a break"]
    else:
        suggestions += ["excited", "calm", "frustrated", "I need help"]

    if "social" in gaps:
        suggestions += ["I love you", "thank you", "yes please", "no thank you", "hug"]
    else:
        suggestions += ["good job", "hello", "bye"]

    if "bathroom" in gaps:
        suggestions += ["I need potty", "bathroom", "help potty", "all done potty"]

    if "bedtime" in gaps:
        suggestions += ["time for bed", "I am sleepy", "good night", "blanket"]

    return list(dict.fromkeys(suggestions))[:12]

def insight_bar_html(title, data, color="#0057B8"):
    items = top_n_from_dict(data, 10)
    if not items:
        return f"<h3>{title}</h3><p>No data yet. Use the board first.</p>"

    max_val = max(v for _, v in items) or 1
    html = f"<h3>{title}</h3><div style='font-family:Arial;'>"
    for label, value in items:
        width = int((value / max_val) * 100)
        html += f"""
        <div style='margin:8px 0;'>
            <div style='font-weight:800;'>{label}: {value}</div>
            <div style='background:#E5E7EB;border-radius:12px;overflow:hidden;'>
                <div style='background:{color};width:{width}%;height:22px;'></div>
            </div>
        </div>
        """
    html += "</div>"
    return html

def generate_parent_summary_text():
    taps = len(PROFILE.get("tap_history", []))
    word_counts = PROFILE.get("word_counts", {})
    phrase_counts = PROFILE.get("phrase_counts", {})
    stats = phrase_length_stats()
    strengths, gaps = identify_strengths_and_gaps()
    suggestions = next_best_words()

    top_words = ", ".join([w for w, c in top_n_from_dict(word_counts, 5)]) or "not enough data yet"
    top_phrases = ", ".join([p for p, c in top_n_from_dict(phrase_counts, 5)]) or "not enough data yet"

    summary = []
    summary.append("TalkFree Parent Insight Summary")
    summary.append("")
    summary.append(f"Total taps: {taps}")
    summary.append(f"Completed spoken sentences: {stats['completed_sentences']}")
    summary.append(f"Average sentence length: {stats['average_length']}")
    summary.append(f"Max sentence length: {stats['max_length']}")
    summary.append("")
    summary.append(f"Most used words: {top_words}")
    summary.append(f"Most used phrases: {top_phrases}")
    summary.append("")
    summary.append("Communication strengths: " + (", ".join(strengths) if strengths else "not enough data yet"))
    summary.append("Growth gaps: " + (", ".join(gaps) if gaps else "none obvious yet"))
    summary.append("")
    summary.append("Next-best vocabulary:")
    for word in suggestions:
        summary.append("- " + word)

    return "\\n".join(summary)

def insight_dashboard_html():
    taps = len(PROFILE.get("tap_history", []))
    word_counts = PROFILE.get("word_counts", {})
    phrase_counts = PROFILE.get("phrase_counts", {})
    stats = phrase_length_stats()
    strengths, gaps = identify_strengths_and_gaps()
    suggestions = next_best_words()
    vocab_scores = vocabulary_score_summary() if "vocabulary_score_summary" in globals() else {}

    html = """
    <div style='background:#111827;color:white;border-radius:22px;padding:18px;margin:12px 0;font-family:Arial;border:4px solid #FFD400;'>
        <div style='font-size:30px;font-weight:900;'>📊 Parent Learning Insights</div>
        <div style='font-size:16px;color:#FDE68A;'>Beta insight layer — reads child board usage data</div>
    </div>
    """

    html += f"""
    <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:10px;font-family:Arial;'>
        <div style='background:#E0F2FE;padding:14px;border-radius:16px;'><b>Total taps</b><br><span style='font-size:28px;font-weight:900;'>{taps}</span></div>
        <div style='background:#DCFCE7;padding:14px;border-radius:16px;'><b>Sentences</b><br><span style='font-size:28px;font-weight:900;'>{stats['completed_sentences']}</span></div>
        <div style='background:#FEF3C7;padding:14px;border-radius:16px;'><b>Avg length</b><br><span style='font-size:28px;font-weight:900;'>{stats['average_length']}</span></div>
        <div style='background:#FCE7F3;padding:14px;border-radius:16px;'><b>Max length</b><br><span style='font-size:28px;font-weight:900;'>{stats['max_length']}</span></div>
    </div>
    """

    html += "<h3>Communication Strengths</h3>"
    html += "<p>" + (", ".join(strengths) if strengths else "Not enough data yet. Let Austin use the board first.") + "</p>"

    html += "<h3>Growth Gaps</h3>"
    html += "<p>" + (", ".join(gaps) if gaps else "No obvious gaps yet.") + "</p>"

    html += "<h3>Next-Best Vocabulary</h3><ul>"
    for word in suggestions:
        html += "<li>" + word + "</li>"
    html += "</ul>"

    html += insight_bar_html("Most Used Words", word_counts, "#0057B8")
    html += insight_bar_html("Most Used Phrases", phrase_counts, "#118C4F")
    html += insight_bar_html("Communication Functions", communication_functions(), "#C2185B")
    html += insight_bar_html("Concept Use", count_by_concept(), "#F57C00")
    html += insight_bar_html("Word Type Use", count_by_word_type(), "#6A1B9A")

    if vocab_scores:
        html += insight_bar_html("Vocabulary Learning Status", vocab_scores, "#424242")

    return html

def save_insight_snapshot():
    PROFILE.setdefault("insight_notes", [])
    PROFILE["insight_notes"].append({
        "time": datetime.now().isoformat(),
        "summary": generate_parent_summary_text()
    })
    save_all()

def add_next_best_words_to_tree():
    global system_message
    suggestions = next_best_words()
    auto_tree_words("\\n".join(suggestions))
    system_message = "Added Insight next-best vocabulary to the tree."
    render_board()



# ============================================================
# Knowledge Object Schema 1.13
# ============================================================

PROFILE.setdefault("schema_version", "knowledge_object_1.13")
PROFILE.setdefault("knowledge_objects", {})

def default_sentence_starters_for_type(word_type):
    if word_type == "food":
        return ["I want", "I want to eat", "I like", "more"]
    if word_type == "feeling":
        return ["I am", "I feel"]
    if word_type == "place":
        return ["I want to go", "I need to go"]
    if word_type == "verb":
        return ["I want to", "I need to", "can I"]
    if word_type == "social":
        return ["I", "yes", "thank you"]
    if word_type == "safety":
        return ["I need", "help", "I feel"]
    if word_type == "person":
        return ["I want", "I love", "help"]
    return ["I want", "I like"]

def object_mastery_score(word):
    count = PROFILE.get("word_counts", {}).get(word, 0)
    if count >= 25:
        return 0.95
    if count >= 10:
        return 0.75
    if count >= 3:
        return 0.45
    return 0.15

def object_learning_status(word):
    explicit = PROFILE.get("learning_status", {}).get(word)
    if explicit:
        return explicit
    score = object_mastery_score(word)
    if score >= 0.9:
        return "mastered"
    if score >= 0.6:
        return "known"
    if score >= 0.25:
        return "learning"
    return "suggested"

def build_knowledge_object(word):
    data = WORDS.get(word, {})
    word_type = data.get("type", "object")
    concept = data.get("concept", word_type)
    rel = PROFILE.get("relationship_graph", {}).get(word, {})
    synonyms = PROFILE.get("synonym_network", {}).get(word, [])

    contexts = rel.get("contexts", [])
    if not contexts:
        for context_name, context_words in PROFILE.get("context_boards", {}).items():
            if word in context_words:
                contexts.append(context_name)

    related = list(dict.fromkeys(rel.get("related", []) + synonyms))

    obj = {
        "id": word.lower().replace(" ", "_").replace("'", ""),
        "name": word,
        "display_text": word,
        "category": rel.get("category", concept),
        "grammar_type": word_type,
        "developmental_stage": int(data.get("stage", 2)),
        "color": TYPE_STYLES.get(word_type, TYPE_STYLES.get("object", {})).get("color", "#00796B"),
        "icon": data.get("image", "🔤"),
        "asset_path": data.get("asset", safe_asset_path(word, word_type)) if "safe_asset_path" in globals() else data.get("asset", ""),
        "contexts": sorted(list(set(contexts))),
        "related_words": related,
        "synonyms": synonyms,
        "opposites": [],
        "sentence_starters": default_sentence_starters_for_type(word_type),
        "prediction_rules": GRAPH.get(word, {}),
        "mastery_score": object_mastery_score(word),
        "learning_status": object_learning_status(word),
        "usage_frequency": PROFILE.get("word_counts", {}).get(word, 0),
        "pronunciation": word,
        "parent_notes": "",
        "learning_goals": [],
        "ai_expansion_rules": {
            "auto_phrase_mining": True,
            "connect_to_category": True,
            "suggest_related_words": True,
            "respect_stage_gate": True
        },
        "free_feature": True,
        "insight_feature": {
            "included_in_reports": True,
            "track_mastery": True,
            "track_context_use": True
        }
    }
    return obj

def rebuild_knowledge_objects():
    objects = {}
    for word in sorted(WORDS.keys()):
        objects[word] = build_knowledge_object(word)
    PROFILE["knowledge_objects"] = objects
    PROFILE["schema_version"] = "knowledge_object_1.13"
    save_all()
    return objects

def validate_knowledge_objects():
    required_fields = [
        "id", "name", "category", "grammar_type", "developmental_stage",
        "color", "icon", "contexts", "related_words", "sentence_starters",
        "mastery_score", "learning_status", "usage_frequency"
    ]
    objects = PROFILE.get("knowledge_objects", {})
    problems = []
    for word, obj in objects.items():
        missing = [field for field in required_fields if field not in obj]
        if missing:
            problems.append({"word": word, "missing": missing})
    return problems

def export_website_schema():
    rebuild_knowledge_objects()
    export = {
        "app": "TalkFree",
        "engine_version": APP_VERSION,
        "schema_version": PROFILE.get("schema_version", "knowledge_object_1.13"),
        "local_first": True,
        "privacy_model": {
            "child_data_local_by_default": True,
            "cloud_sync_optional": True,
            "insight_reads_profile": True
        },
        "free_features": [
            "AAC board",
            "next-word prediction",
            "local child profile",
            "add/edit/delete words",
            "bulk import",
            "auto-tree builder",
            "context boards",
            "basic vocabulary suggestions"
        ],
        "paid_insight_features": [
            "progress dashboards",
            "strengths and gaps",
            "trend analysis",
            "therapist summaries",
            "exportable reports",
            "advanced learning recommendations"
        ],
        "type_styles": TYPE_STYLES,
        "stage_boards": STAGE_BOARDS,
        "language_graph": GRAPH,
        "knowledge_objects": PROFILE.get("knowledge_objects", {}),
        "context_boards": PROFILE.get("context_boards", {}),
        "developmental_maps": PROFILE.get("developmental_maps", {}),
        "concept_network": PROFILE.get("concept_network", {}),
        "relationship_graph": PROFILE.get("relationship_graph", {}),
        "profile_template": {
            "selected_stage": PROFILE.get("selected_stage", "Stage 2: Sentence Builder"),
            "active_context": PROFILE.get("active_context", "Home"),
            "word_counts": {},
            "phrase_counts": {},
            "tap_history": [],
            "sentence_history": [],
            "learning_status": {}
        }
    }
    return export

def save_website_schema_file():
    export = export_website_schema()
    path = DATA_DIR / "talkfree_website_schema.json"
    path.write_text(json.dumps(export, indent=2), encoding="utf-8")
    return str(path)

def knowledge_object_status_html():
    rebuild_knowledge_objects()
    objects = PROFILE.get("knowledge_objects", {})
    problems = validate_knowledge_objects()
    sample_names = list(objects.keys())[:8]

    html = "<h3>Knowledge Object Schema</h3>"
    html += "<ul>"
    html += f"<li>Total objects: {len(objects)}</li>"
    html += f"<li>Validation problems: {len(problems)}</li>"
    html += f"<li>Schema version: {PROFILE.get('schema_version')}</li>"
    html += "</ul>"
    html += "<h4>Sample Objects</h4><ul>"
    for name in sample_names:
        obj = objects[name]
        html += f"<li><b>{name}</b> — {obj.get('category')} / {obj.get('grammar_type')} / Stage {obj.get('developmental_stage')}</li>"
    html += "</ul>"
    if problems:
        html += "<h4>Problems</h4><pre>" + json.dumps(problems[:10], indent=2) + "</pre>"
    return html

rebuild_knowledge_objects()


def validate_stage_leaks():
    leaks = {}
    for stage_name, words in STAGE_BOARDS.items():
        allowed = 1 if stage_name.startswith("Stage 1") else 2 if stage_name.startswith("Stage 2") else 3
        bad = [w for w in words if int(WORDS.get(w, {}).get("stage", 2)) > allowed]
        if bad:
            leaks[stage_name] = bad
    return leaks

repair_core_data()
repair_core_data()
save_all()
print("TalkFreeEngine_1.13 setup complete. Run the board cell.")


TalkFreeEngine_1.13 setup complete. Run the board cell.


In [2]:

# TalkFreeEngine_1.7 — Touchscreen Board

def render_board():
    clear_output(wait=True)

    phrase = current_phrase()
    stage = selected_stage()
    sentence_display = phrase if phrase else "&nbsp;"

    css = widgets.HTML("""
    <style>
    .widget-button {
        font-size: 30px !important;
        font-weight: 900 !important;
        border-radius: 28px !important;
        border: 5px solid rgba(255,255,255,0.75) !important;
        color: white !important;
        box-shadow: 0 7px 0 rgba(0,0,0,0.35) !important;
        white-space: pre-line !important;
        line-height: 1.15 !important;
    }
    .widget-button:active { transform: scale(0.97); }
    </style>
    """)

    title = widgets.HTML(f"""
    <div style='background:#111827;color:white;padding:18px 24px;border-radius:26px;margin-bottom:14px;border:5px solid #FFD400;box-shadow:0 5px 0 rgba(0,0,0,.40);font-family:Arial;'>
        <div style='font-size:40px;font-weight:900;'>🗣️ {APP_VERSION}</div>
        <div style='font-size:18px;color:#FDE68A;'>knowledge objects • website-ready schema • shared data</div>
    </div>
    """)

    sentence_box = widgets.HTML(
        "<div style='background:#111;color:#fff;border:7px solid #FFD400;border-radius:28px;padding:28px;min-height:90px;font-size:58px;font-weight:900;box-shadow:0 7px 0 rgba(0,0,0,.45);margin-bottom:18px;font-family:Arial;'>"
        + sentence_display +
        "</div>"
    )

    controls = []
    for label, action in [("🔊 Speak", speak_sentence), ("⬅️ Back", backspace), ("❌ Clear", clear_sentence), ("⚙️ Parent", toggle_parent_tools)]:
        btn = widgets.Button(description=label, layout=widgets.Layout(width="24%", height="74px"))
        btn.on_click(lambda b, a=action: a())
        controls.append(btn)

    stage_picker = widgets.Dropdown(options=list(STAGE_BOARDS.keys()), value=stage, description="Stage:", layout=widgets.Layout(width="70%"))
    stage_btn = widgets.Button(description="Apply", layout=widgets.Layout(width="25%", height="48px"))
    stage_btn.on_click(lambda b: change_stage(stage_picker.value))

    board_words = get_predictions()
    board_title = widgets.HTML(f"<div style='font-family:Arial;color:#111827;font-size:30px;font-weight:900;margin:16px 0 10px 0;'>{stage if not phrase else 'Next words after: ' + phrase}</div>")

    cols = 3 if stage.startswith("Stage 1") else 4
    width = "32%" if cols == 3 else "24%"
    height = "190px" if cols == 3 else "170px"

    rows = []
    for i in range(0, len(board_words), cols):
        row = []
        for word in board_words[i:i+cols]:
            data = WORDS.get(word, {"image": "🔤", "type": "utility"})
            style = TYPE_STYLES.get(data.get("type", "utility"), TYPE_STYLES["utility"])
            label = f"{data.get('image','🔤')}\n{word.upper()}"
            btn = widgets.Button(description=label, layout=widgets.Layout(width=width, height=height))
            btn.style.button_color = style["color"]
            btn.on_click(lambda b, w=word: add_word(w))
            row.append(btn)
        rows.append(widgets.HBox(row))

    items = [css, title, sentence_box, widgets.HBox(controls), widgets.HBox([stage_picker, stage_btn]), board_title, *rows]

    if parent_tools_open:
        status_html = f"<div style='background:#FEF3C7;color:#111827;border-radius:12px;padding:10px;margin:8px 0;font-weight:800;'>{system_message}</div>" if system_message else ""

        custom_word = widgets.Text(placeholder="Example: milk", description="Word:", layout=widgets.Layout(width="38%"))
        custom_type = widgets.Dropdown(options=list(TYPE_STYLES.keys()), value="food", description="Type:", layout=widgets.Layout(width="25%"))
        custom_image = widgets.Text(placeholder="Emoji now, image later", description="Image:", layout=widgets.Layout(width="25%"))
        show_after = widgets.Text(value=current_phrase() if current_phrase() else "I want", placeholder="Example: I want", description="Show after:", layout=widgets.Layout(width="60%"))

        add_word_btn = widgets.Button(description="Add / Update Word", layout=widgets.Layout(width="30%", height="50px"))
        add_word_btn.on_click(lambda b: add_custom_word(custom_word.value, custom_type.value, custom_image.value, show_after.value))

        delete_picker = widgets.Dropdown(options=sorted(list(WORDS.keys())), description="Delete:", layout=widgets.Layout(width="55%"))
        delete_btn = widgets.Button(description="Delete Word", layout=widgets.Layout(width="22%", height="50px"))
        undo_btn = widgets.Button(description="Undo Delete", layout=widgets.Layout(width="22%", height="50px"))
        delete_btn.on_click(lambda b: delete_word(delete_picker.value))
        undo_btn.on_click(lambda b: undo_delete_word())

        status = {"words": len(WORDS), "graph_paths": len(GRAPH), "stage": stage, "current_phrase": current_phrase(), "stage_leaks": validate_stage_leaks()}
        status_box = widgets.HTML("<details><summary>Engine Status</summary><pre>" + json.dumps(status, indent=2) + "</pre></details>")

        context_picker = widgets.Dropdown(options=list(CONTEXT_BOARDS.keys()), value=PROFILE.get("active_context", "Home"), description="Context:", layout=widgets.Layout(width="55%"))
        context_btn = widgets.Button(description="Apply Context", layout=widgets.Layout(width="30%", height="50px"))
        context_btn.on_click(lambda b: change_context(context_picker.value))

        genome_btn = widgets.Button(description="Show Genome Status", layout=widgets.Layout(width="30%", height="50px"))
        genome_output = widgets.Output()
        def _show_genome(b):
            with genome_output:
                clear_output(wait=True)
                display(widgets.HTML(genome_status_html()))
        genome_btn.on_click(_show_genome)

        goal_btn = widgets.Button(description="Add Goal Suggestions", layout=widgets.Layout(width="30%", height="50px"))
        goal_btn.on_click(lambda b: add_goal_suggestions())

        schema_btn = widgets.Button(description="Show Object Schema", layout=widgets.Layout(width="30%", height="50px"))
        schema_output = widgets.Output()
        def _show_schema(b):
            with schema_output:
                clear_output(wait=True)
                display(widgets.HTML(knowledge_object_status_html()))
        schema_btn.on_click(_show_schema)

        export_schema_btn = widgets.Button(description="Export Website JSON", layout=widgets.Layout(width="30%", height="50px"))
        schema_file_output = widgets.Output()
        def _export_schema(b):
            path = save_website_schema_file()
            with schema_file_output:
                clear_output(wait=True)
                display(widgets.HTML("<b>Saved:</b> " + path))
        export_schema_btn.on_click(_export_schema)

        insight_btn = widgets.Button(description="Open Parent Insights", layout=widgets.Layout(width="30%", height="50px"))
        insight_output = widgets.Output()
        def _open_insights(b):
            with insight_output:
                clear_output(wait=True)
                display(widgets.HTML(insight_dashboard_html()))
        insight_btn.on_click(_open_insights)

        export_btn = widgets.Button(description="Export Summary", layout=widgets.Layout(width="30%", height="50px"))
        export_output = widgets.Output()
        def _export_summary(b):
            save_insight_snapshot()
            with export_output:
                clear_output(wait=True)
                display(widgets.Textarea(value=generate_parent_summary_text(), layout=widgets.Layout(width="95%", height="260px")))
        export_btn.on_click(_export_summary)

        add_insight_words_btn = widgets.Button(description="Add Insight Words", layout=widgets.Layout(width="30%", height="50px"))
        add_insight_words_btn.on_click(lambda b: add_next_best_words_to_tree())

        phrase_text = widgets.Text(value="I love you", description="Phrase:", layout=widgets.Layout(width="55%"))
        phrase_btn = widgets.Button(description="Add Phrase Chain", layout=widgets.Layout(width="30%", height="50px"))
        phrase_btn.on_click(lambda b: add_phrase_chain(phrase_text.value, stage=1, image="💗", word_type="social"))

        test_btn = widgets.Button(description="Run Self-Tests", layout=widgets.Layout(width="30%", height="50px"))
        test_output = widgets.Output()
        def _run_tests(b):
            with test_output:
                clear_output(wait=True)
                display(widgets.HTML(run_engine_self_tests()))
        test_btn.on_click(_run_tests)

        reset_btn = widgets.Button(description="Repair Austin Baseline", layout=widgets.Layout(width="30%", height="50px"))
        reset_btn.on_click(lambda b: (repair_core_data(), render_board()))

        auto_words = widgets.Textarea(
            value="apple juice\nwater\nI\nwant\nyes\nno\nplease\nthankyou\nand\nawesome\ngood\nokay\ntime\nfor\nbed\ngummies\nfrench fries\nchicken nuggets\nI'm hungry\nI'm thirsty",
            placeholder="Paste words or phrases, one per line.",
            description="Auto-tree:",
            layout=widgets.Layout(width="95%", height="180px")
        )
        auto_btn = widgets.Button(description="Build Auto-Tree From Words", layout=widgets.Layout(width="40%", height="54px"))
        auto_btn.on_click(lambda b: auto_tree_words(auto_words.value))

        growth_box = widgets.Textarea(
            value="\n".join(vocabulary_growth_pack_for_austin()),
            description="Growth Pack:",
            layout=widgets.Layout(width="95%", height="160px")
        )
        growth_btn = widgets.Button(description="Add Growth Pack", layout=widgets.Layout(width="40%", height="54px"))
        growth_btn.on_click(lambda b: auto_tree_words(growth_box.value))

        items.extend([
            widgets.HTML("<div style='background:#1F2937;color:white;border-radius:20px;padding:16px;margin-top:18px;font-family:Arial;'><div style='font-size:26px;font-weight:900;'>Parent Tools</div><div>Add word + choose where it appears, or paste a group and let Auto-Tree place it.</div></div>"),
            widgets.HTML(status_html),
            widgets.HTML("<h3>Engine Tools</h3>"),
            widgets.HBox([test_btn, reset_btn, genome_btn]),
            test_output,
            genome_output,
            widgets.HTML("<h3>Website-Ready Engine Schema</h3>"),
            widgets.HBox([schema_btn, export_schema_btn]),
            schema_output,
            schema_file_output,
            widgets.HTML("<h3>Parent Learning Insights</h3>"),
            widgets.HBox([insight_btn, export_btn, add_insight_words_btn]),
            insight_output,
            export_output,
            widgets.HTML("<h3>Context Mode</h3>"),
            widgets.HBox([context_picker, context_btn, goal_btn]),
            widgets.HTML("<h3>Add Full Phrase Chain</h3>"),
            widgets.HBox([phrase_text, phrase_btn]),
            widgets.HTML("<h3>Beta Auto-Tree Builder</h3>"),
            auto_words,
            auto_btn,
            widgets.HTML("<h3>Austin Vocabulary Growth Pack</h3>"),
            growth_box,
            growth_btn,
            widgets.HTML("<h3>Add / Update Single Word</h3>"),
            widgets.HBox([custom_word, custom_type, custom_image]),
            widgets.HBox([show_after, add_word_btn]),
            widgets.HTML("<h3>Delete / Undo</h3>"),
            widgets.HBox([delete_picker, delete_btn, undo_btn]),
            status_box
        ])

    display(*items)

render_board()


HTML(value='\n    <style>\n    .widget-button {\n        font-size: 30px !important;\n        font-weight: 900…

HTML(value="\n    <div style='background:#111827;color:white;padding:18px 24px;border-radius:26px;margin-botto…

HTML(value="<div style='background:#111;color:#fff;border:7px solid #FFD400;border-radius:28px;padding:28px;mi…

HTML(value="<div style='font-family:Arial;color:#111827;font-size:30px;font-weight:900;margin:16px 0 10px 0;'>…

## TalkFreeEngine_1.13

### What changed

This version adds the portable object schema the future website needs.

Every word now becomes a Knowledge Object with:

- name
- category
- grammar type
- stage
- color
- icon
- asset path
- contexts
- related words
- synonyms
- sentence starters
- prediction rules
- mastery score
- usage frequency
- pronunciation
- parent notes
- learning goals
- AI expansion rules

### Website Readiness

Parent Tools now includes:

- Show Object Schema
- Export Website JSON

That JSON is the bridge from Colab into GitHub / website.
